In [0]:
%pip install shapely

since the bronze data was read in multiple batches, it's good to optimize the delta table before using it as a spark table

In [0]:
spark.sql("OPTIMIZE ca_healthcare_fac_bronze.ca_general_land_use_data.ca_land_use_df")

In [0]:
import pandas as pd
# from arcgis.features import GeoAccessor, GeoSeriesAccessor

ca_land_bronze_path = "ca_healthcare_fac_bronze.ca_general_land_use_data.ca_land_use_df"

# read the entire bronze table table first then filter for open space and public land
ca_land_bronze_df = spark.read.table(ca_land_bronze_path)

ca_land_bronze_filtered = ca_land_bronze_df.filter(ca_land_bronze_df.ucd_description == "Open space and public lands")

ca_land_public_space_pddf = ca_land_bronze_filtered.toPandas()

ca_land_public_space_pddf.shape

In [0]:
ca_land_public_space_pddf.head()

In [0]:
ca_land_public_space_pddf['geometry_rings'][0]

In [0]:
from shapely.geometry import Polygon

def get_centroid(rings):
    ring = rings[0]
    polygon = Polygon(ring)
    return polygon.centroid.x, polygon.centroid.y

ca_land_public_space_pddf[['centroid_lon', 'centroid_lat']] = ca_land_public_space_pddf['geometry_rings'].apply(lambda rings: pd.Series(get_centroid(rings=rings)))
ca_land_public_space_pddf.head()




Lookup the centroid latidue and longitude and get the mssa object